In [1]:
from typing import Optional
from pydantic import BaseModel, Field

from langchain_core.output_parsers import PydanticOutputParser
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

/home/greg/Documents/Estudos/CURSO_ASIMOV/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_community.cache import SQLiteCache
from langchain_core.globals import set_llm_cache
from os import path
from tempfile import gettempdir



# set_llm_cache(SQLiteCache(path.join(gettempdir(), "langchain_cache.db")))

In [3]:
chat_model = Ollama(model="minimax-m2.5:cloud", temperature=.6)

/tmp/ipykernel_90077/1282249068.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  chat_model = Ollama(model="minimax-m2.5:cloud", temperature=.6)


In [4]:
review_cliente = """Este soprador de folhas é bastante incrível. Ele tem 
quatro configurações: sopro de vela, brisa suave, cidade ventosa 
e tornado. Chegou em dois dias, bem a tempo para o presente de 
aniversário da minha esposa. Acho que minha esposa gostou tanto 
que ficou sem palavras. Até agora, fui o único a usá-lo, e tenho 
usado em todas as manhãs alternadas para limpar as folhas do 
nosso gramado. É um pouco mais caro do que os outros sopradores 
de folhas disponíveis no mercado, mas acho que vale a pena pelas 
características extras."""

In [5]:
from pydantic import field_validator

class ProductReview(BaseModel):
    """Avalia Review do Cliente"""     
    produto: Optional[str] = Field(default=None, description="Breve descrição do produto, máximo de seis palavras")      
    entrega_satisfacao: Optional[bool] = Field(default=None, description="O cliente está satisfeito com a entrega? true ou false")      
    produto_satisfacao: Optional[bool] = Field(default=None, description="O cliente está satisfeito com o produto? true ou false")      
    atendimento: Optional[str] = Field(default=None, description="Satisfação com atendimento, máximo 8 palavras (sempre string)")     
    satisfacao: Optional[str] = Field(default=None, description="Satisfação com a compra em poucas palavras (sempre string)")
    
    @field_validator("produto", "atendimento", "satisfacao", mode="before")
    @classmethod
    def garantir_string(cls, v):
        """Se o LLM devolver objeto/dict, extrai texto; senão garante str ou None."""
        if v is None:
            return None
        if isinstance(v, dict):
            return v.get("nota") or v.get("descricao") or " ".join(str(x) for x in v.values())
        return str(v) if not isinstance(v, str) else v
    
    @field_validator("entrega_satisfacao", "produto_satisfacao", mode="before")
    @classmethod
    def garantir_bool(cls, v):
        """Converte 'Sim'/'Não' ou string em bool; mantém None."""
        if v is None:
            return None
        if isinstance(v, bool):
            return v
        s = str(v).strip().lower()
        if s in ("sim", "satisfeito", "yes", "true", "1"):
            return True
        if s in ("não", "nao", "insatisfeito", "no", "false", "0"):
            return False
        return None
    
    def __str__(self):
        return f"Produto: {self.produto} Entrega: {'Sim' if self.entrega_satisfacao else 'Não'} Produto: {'Sim' if self.produto_satisfacao else 'Não'} Atendimento: {self.atendimento} Satisfação: {self.satisfacao}\n"
    
    def __repr__(self):
        return f"Produto: {self.produto} Entrega: {self.entrega_satisfacao} Produto: {self.produto_satisfacao} Atendimento: {self.atendimento} Satisfação: {self.satisfacao}\n"

In [6]:
tradutor = PydanticOutputParser(pydantic_object=ProductReview)

In [7]:
reviews_do_cliente = [
"Comprei o soprador de ar portátil para limpar a garagem. Chegou dois dias antes do prazo, bem embalado. O atendimento respondeu minhas dúvidas rapidamente. O produto é forte, leve e muito prático. Fiquei muito satisfeito com a compra.",
"O soprador de ar elétrico chegou exatamente no prazo informado. A caixa veio intacta e a entrega foi tranquila. O atendimento foi educado, mas um pouco demorado. O aparelho funciona bem para folhas secas e poeira. No geral, estou satisfeito.",
"Recebi o soprador com três dias de atraso e isso me incomodou bastante. O suporte pediu desculpas, porém demorou para responder. O produto até funciona, mas a potência é menor do que eu esperava. Minha experiência foi apenas razoável.",
"Adorei esse soprador de ar compacto. A entrega foi super rápida e o entregador foi cuidadoso. O atendimento da loja foi excelente do início ao fim. O equipamento é silencioso, eficiente e fácil de guardar. Compra muito boa.",
"O soprador chegou no prazo, mas a embalagem veio amassada. Entrei em contato e o atendimento resolveu tudo com atenção. O produto funciona corretamente, embora seja um pouco barulhento. Mesmo assim, fiquei satisfeito com o resultado.",
"Infelizmente o soprador de ar veio com defeito e não ligava. A entrega foi rápida, então não tive problema com essa parte. O atendimento foi prestativo e iniciou a troca sem burocracia. Como o produto falhou, minha satisfação com a compra foi baixa.",
"Produto excelente para uso doméstico. O soprador chegou antes do previsto, bem protegido e sem avarias. O suporte foi muito cordial e tirou minhas dúvidas sobre voltagem. A potência é ótima para quintal e varanda. Recomendo bastante.",
"O soprador de ar é bonito e aparenta boa durabilidade. A entrega atrasou um dia, mas nada grave. O atendimento foi ok, sem ser excelente. No uso, ele limpa bem cantos e folhas leves. Estou moderadamente satisfeito.",
"Chegou muito rápido e em perfeitas condições. O atendimento foi ágil e simpático no chat. O soprador é mais potente do que imaginei e facilitou muito a limpeza do carro. Estou muito feliz com a compra.",
"Não gostei da experiência. O soprador demorou quase uma semana além do prazo e ninguém informava nada. Quando falei com o atendimento, a resposta foi fria e pouco útil. O produto funciona, mas é fraco. Não compraria novamente.",
"Esse soprador de ar manual me surpreendeu positivamente. A entrega foi pontual e a embalagem estava impecável. O atendimento foi rápido e resolveu uma dúvida sobre garantia. O produto é simples, mas entrega exatamente o que promete. Estou satisfeito.",
"Recebi o produto antes do prazo e isso foi ótimo. O atendimento da loja foi excelente, muito educado e eficiente. O soprador é leve, potente e fácil de manusear no dia a dia. Minha satisfação com a compra foi muito alta.",
"O soprador de ar chegou no prazo certo, porém com a caixa mal fechada. O atendimento demorou um pouco, mas foi educado. O aparelho funciona bem para poeira, mas não tem tanta força para folhas molhadas. Considero uma compra razoável.",
"Gostei muito do produto. A entrega foi rápida e sem nenhum problema. O atendimento foi atencioso desde a confirmação do pedido até o pós-venda. O soprador tem ótima potência e acabamento. Foi uma excelente compra.",
"Minha experiência foi ruim. O produto chegou atrasado, com marcas de uso, e precisei reclamar. O atendimento respondeu, mas sem muita disposição para ajudar. O soprador funciona de forma instável e perde força às vezes. Fiquei decepcionado.",
"Entrega perfeita, chegou no dia combinado e muito bem embalado. Atendimento educado e eficiente pelo WhatsApp. O soprador de ar é compacto, resistente e muito útil para limpar computador e bancada. Estou bastante satisfeito.",
"O soprador é bom, mas eu esperava mais potência pelo preço. A entrega ocorreu no prazo e sem problemas. O atendimento foi cordial, embora não tenha sido muito rápido. Para uso leve ele atende bem. Minha satisfação é mediana.",
"Produto excelente e fácil de usar. A entrega foi super rápida, chegou no dia seguinte. O atendimento foi ótimo, com respostas claras e rápidas. O soprador resolveu exatamente o que eu precisava na oficina. Compra aprovada.",
"Chegou atrasado e isso prejudicou bastante minha avaliação. O atendimento pediu desculpas e acompanhou o envio até a entrega, o que ajudou. O soprador em si é muito bom, potente e resistente. No fim, fiquei satisfeito com o produto, mas não com a entrega.",
"Gostei do soprador de ar porque ele é leve e não cansa a mão. A entrega foi pontual e veio tudo certo. O atendimento foi bom, tirou minhas dúvidas sem enrolação. O desempenho é ótimo para uso residencial. Estou satisfeito com a compra.",
"O produto veio certo, mas a entrega demorou mais do que o prometido. O atendimento só respondeu depois de dois dias, o que foi frustrante. O soprador funciona bem e tem boa potência. Minha satisfação geral ficou no meio termo.",
"Adorei a compra. O soprador chegou muito antes do prazo e bem protegido. O atendimento foi excelente e muito gentil. O equipamento é forte, prático e ajudou muito na limpeza do quintal. Estou plenamente satisfeito.",
"Não recomendo. A entrega foi rápida, mas o soprador veio com peça solta. O atendimento foi confuso e demorado para resolver. Além disso, o produto parece frágil e tem baixa potência. Experiência ruim.",
"Recebi o soprador dentro do prazo e sem danos. O atendimento foi normal, nada extraordinário. O produto cumpre o que promete para sujeira leve e poeira fina. Considero uma compra honesta e satisfatória.",
"Produto muito bom para limpar folhas secas e poeira da garagem. A entrega foi rápida e a embalagem veio reforçada. O atendimento foi atencioso e educado. Estou satisfeito tanto com o produto quanto com a compra.",
"Chegou no prazo, mas a caixa estava molhada. O atendimento foi excelente e confirmou que poderia trocar se houvesse dano. Felizmente o soprador estava funcionando bem e com boa potência. No final, fiquei satisfeito.",
"O soprador de ar portátil é fácil de guardar e simples de usar. A entrega foi rápida e sem intercorrências. O atendimento foi apenas razoável, demorou para enviar a nota fiscal. O produto, porém, me agradou bastante. Compra boa.",
"Infelizmente o produto não correspondeu ao anúncio. A entrega foi no prazo, mas o soprador é fraco e esquenta rápido. O atendimento foi educado, porém não resolveu minha insatisfação. Saí decepcionado com a compra.",
"Excelente experiência. O soprador chegou antes do prazo, bem embalado e com manual. O atendimento foi rápido, simpático e muito eficiente. O produto é potente, silencioso e parece durável. Estou muito satisfeito.",
"O atendimento foi um dos melhores que já tive em compra online. A entrega ocorreu dentro do prazo e sem problemas. O soprador funciona muito bem, principalmente em espaços pequenos. Satisfação alta com a compra.",
"Recebi com atraso e sem atualização de rastreio. O atendimento demorou, mas quando respondeu foi bastante educado. O soprador de ar é bom, tem força adequada e acabamento decente. Apesar do transtorno na entrega, gostei do produto.",
"Produto chegou no dia certo e bem protegido. O atendimento foi eficiente para alterar o endereço antes do envio. O soprador é resistente, fácil de montar e tem ótimo desempenho. Fiquei muito satisfeito.",
"Não fiquei feliz com a compra. A entrega foi rápida, mas o soprador faz barulho excessivo e vibra muito. O atendimento foi correto, sem grande esforço para ajudar. Minha satisfação final foi baixa.",
"Gostei bastante do soprador de ar. A entrega foi pontual e o produto veio lacrado. O atendimento foi cordial e objetivo. Ele limpa bem folhas, poeira e sujeira leve. Estou satisfeito com o custo-benefício.",
"Entrega super rápida, chegou em menos de 48 horas. O atendimento foi ótimo e acompanhou meu pedido. O soprador tem boa potência e ótima ergonomia. Compra excelente, valeu a pena.",
"O produto é apenas mediano. A entrega aconteceu no prazo, sem problemas. O atendimento foi bom, mas um pouco automático. O soprador funciona, porém não tem a força que eu precisava. Minha satisfação é razoável.",
"Recebi o soprador com atraso de quatro dias. O atendimento foi muito bom e sempre respondeu com educação. O produto chegou em perfeito estado e apresentou ótimo desempenho. Fiquei satisfeito com o produto, mas a entrega deixou a desejar.",
"Comprei para uso no escritório e gostei muito. A entrega foi rápida e a embalagem veio bem protegida. O atendimento foi excelente e esclareceu minhas dúvidas técnicas. O soprador é eficiente, compacto e silencioso. Estou muito satisfeito.",
"Não gostei do atendimento, achei seco e demorado. A entrega, por outro lado, foi rápida e organizada. O soprador de ar funciona muito bem e tem excelente potência. A compra foi boa, mas o suporte precisa melhorar.",
"O soprador chegou no prazo correto e em boas condições. O atendimento foi prestativo e resolveu um erro no cadastro rapidamente. O produto é robusto, eficiente e fácil de usar. Minha satisfação com a compra é alta.",
"Produto excelente para quem quer praticidade. A entrega foi antes do prazo, muito bem feita. O atendimento foi rápido, simpático e eficiente. O soprador superou minhas expectativas no desempenho. Compra ótima.",
"Recebi o produto no prazo, mas com manual faltando. O atendimento enviou o PDF rapidamente e foi muito cordial. O soprador é bom, tem força suficiente e parece resistente. Fiquei satisfeito no geral.",
"A entrega atrasou bastante e quase cancelei o pedido. O atendimento foi confuso no começo, depois melhorou. Quando o soprador chegou, vi que ele é potente e de boa qualidade. Minha satisfação com o produto é boa, mas a experiência geral foi cansativa.",
"Gostei muito da compra. O soprador chegou rápido, bem embalado e funcionando perfeitamente. O atendimento foi excelente durante todo o processo. É um produto prático, potente e com ótimo acabamento. Muito satisfeito.",
"Não valeu a pena para mim. A entrega foi no prazo, mas o soprador tem potência fraca e acabamento simples. O atendimento foi educado, porém não resolveu minhas dúvidas técnicas. Fiquei insatisfeito com a compra.",
"Produto bom e entrega correta. O atendimento foi gentil e respondeu no mesmo dia. O soprador faz um pouco de barulho, mas limpa bem e é fácil de usar. No geral, estou satisfeito.",
"Excelente produto para limpeza rápida. A entrega foi muito ágil e chegou bem embalada. O atendimento foi impecável e muito atencioso. O soprador é forte, leve e eficiente. Compra muito satisfatória.",
"A entrega foi rápida, mas recebi a cor errada. O atendimento resolveu o problema com educação e agilidade. O soprador funciona bem e atende minhas necessidades. Apesar do erro inicial, fiquei satisfeito.",
"Comprei sem muita expectativa e fui surpreendido. A entrega foi no prazo e sem complicações. O atendimento foi muito bom, com respostas claras. O soprador de ar tem ótima potência e é bem construído. Estou bastante satisfeito.",
"Experiência excelente do começo ao fim. O soprador chegou antes do prazo, muito bem embalado e sem defeitos. O atendimento foi rápido, cordial e eficiente. O produto é potente, prático e de ótima qualidade. Estou totalmente satisfeito com a compra."
]

In [8]:
def avaliar_review(review_do_cliente: list[str]) -> list[ProductReview]:
    respostas_estruturada_classificadas: list[ProductReview] = []
    for review in review_do_cliente:
        print(review)

        prompt = ChatPromptTemplate.from_messages([
                SystemMessage("Você é um analista de produtos. Extraia as informações do texto do cliente. "
                        "REGRAS: produto, atendimento e satisfacao são STRING (texto entre aspas). entrega_satisfacao e produto_satisfacao são BOOLEAN (true ou false). "
                        "Use APENAS os campos: produto, entrega_satisfacao, produto_satisfacao, atendimento, satisfacao. "
                        "Responda APENAS com um objeto JSON, sem texto antes ou depois:\n{formato_exigido}"),
                HumanMessage(f"Analise o seguinte review e extraia as informações no formato JSON:\n\n{review}")])        
        
        corrente = prompt | chat_model | tradutor
        resposta_estruturada = corrente.invoke({
                "texto_do_cliente": review,
                "formato_exigido": tradutor.get_format_instructions()
            })
        respostas_estruturada_classificadas.append(resposta_estruturada)
    return respostas_estruturada_classificadas

In [10]:
review_avaliados = avaliar_review(reviews_do_cliente)

Comprei o soprador de ar portátil para limpar a garagem. Chegou dois dias antes do prazo, bem embalado. O atendimento respondeu minhas dúvidas rapidamente. O produto é forte, leve e muito prático. Fiquei muito satisfeito com a compra.
O soprador de ar elétrico chegou exatamente no prazo informado. A caixa veio intacta e a entrega foi tranquila. O atendimento foi educado, mas um pouco demorado. O aparelho funciona bem para folhas secas e poeira. No geral, estou satisfeito.
Recebi o soprador com três dias de atraso e isso me incomodou bastante. O suporte pediu desculpas, porém demorou para responder. O produto até funciona, mas a potência é menor do que eu esperava. Minha experiência foi apenas razoável.
Adorei esse soprador de ar compacto. A entrega foi super rápida e o entregador foi cuidadoso. O atendimento da loja foi excelente do início ao fim. O equipamento é silencioso, eficiente e fácil de guardar. Compra muito boa.
O soprador chegou no prazo, mas a embalagem veio amassada. Entre

In [11]:
for produto in review_avaliados:
    if not produto.produto_satisfacao:
        print(produto)

Produto: soprador Entrega: Não Produto: Não Atendimento: suporte pediu desculpas, porém demorou para responder Satisfação: razoável

Produto: soprador de ar Entrega: Sim Produto: Não Atendimento: prestativo Satisfação: baixa

Produto: soprador Entrega: Não Produto: Não Atendimento: fria e pouco útil Satisfação: Não gostei da experiência

Produto: soprador de ar Entrega: Sim Produto: Não Atendimento: demorou um pouco, mas foi educado Satisfação: razoável

Produto: soprador Entrega: Não Produto: Não Atendimento: sem disposição para ajudar Satisfação: decepcionado

Produto: soprador Entrega: Sim Produto: Não Atendimento: confuso e demorado Satisfação: ruim

Produto: soprador Entrega: Sim Produto: Não Atendimento: educado porém não resolveu Satisfação: decepcionado

Produto: soprador Entrega: Sim Produto: Não Atendimento: correto, sem grande esforço para ajudar Satisfação: baixa

Produto: soprador Entrega: Sim Produto: Não Atendimento: bom, mas um pouco automático Satisfação: razoável

Pro

In [12]:
for produto in review_avaliados:
    if produto.produto_satisfacao:
        print(produto)

Produto: soprador de ar portátil Entrega: Sim Produto: Sim Atendimento: respondeu minhas dúvidas rapidamente Satisfação: muito satisfeito com a compra

Produto: soprador de ar elétrico Entrega: Sim Produto: Sim Atendimento: educado, mas um pouco demorado Satisfação: satisfeito

Produto: soprador de ar compacto Entrega: Sim Produto: Sim Atendimento: excelente Satisfação: muito boa

Produto: soprador Entrega: Não Produto: Sim Atendimento: resolveu tudo com atenção Satisfação: True

Produto: soprador Entrega: Sim Produto: Sim Atendimento: suporte muito cordial Satisfação: recomendo bastante

Produto: soprador de ar Entrega: Não Produto: Sim Atendimento: ok Satisfação: moderadamente satisfeito

Produto: soprador Entrega: Sim Produto: Sim Atendimento: ágil e simpático Satisfação: muito feliz

Produto: soprador de ar manual Entrega: Sim Produto: Sim Atendimento: rápido e resolveu uma dúvida sobre garantia Satisfação: satisfeito

Produto: soprador Entrega: Sim Produto: Sim Atendimento: excele

In [13]:
for produto in review_avaliados:
    if produto.produto_satisfacao is None:
        print(produto)